In [1]:
seed_cue_csv = "/data/gpfs/projects/punim2219/LM_with_SWOW/sukaih/cultural-lexis-finetune-llms/notebooks/Simon-1-Dataset-Generation/cues_seed.csv"

root_dataset_dir = "/data/gpfs/projects/punim2219/LM_with_SWOW/sukaih/cultural-lexis-finetune-llms/data/03_primary/participant_swow_collection_us"

meta_info_json_fp = "/data/gpfs/projects/punim2219/LM_with_SWOW/sukaih/cultural-lexis-finetune-llms/data/03_primary/participant_swow_collection_us/participant_swow_collection_us_seed_43/cue_meta_info.jsonl"


In [2]:
import os
import pandas as pd
import json
os.environ['workspaceFolder'] = "/data/gpfs/projects/punim2219/LM_with_SWOW/sukaih/cultural-lexis-finetune-llms"
os.environ['WORKING_DIR'] = "/data/gpfs/projects/punim2219/LM_with_SWOW/sukaih/cultural-lexis-finetune-llms"
os.environ['PYTHONPATH'] = "/data/gpfs/projects/punim2219/LM_with_SWOW/sukaih/cultural-lexis-finetune-llms"
os.environ['HF_HOME'] = "/data/gpfs/projects/punim2219/LM_with_SWOW/sukaih/cultural-lexis-finetune-llms/../huggingface"
os.environ['TRITON_CACHE_DIR'] = "/data/gpfs/projects/punim2219/LM_with_SWOW/sukaih/cultural-lexis-finetune-llms/../triton_cache"

In [3]:
cue_meta_info_dict = dict()
for seed_id in [43,44,45,46,47]:
    cue_meta_info = pd.read_json(f'/data/gpfs/projects/punim2219/LM_with_SWOW/sukaih/cultural-lexis-finetune-llms/data/03_primary/participant_swow_collection_us/participant_swow_collection_us_seed_{seed_id}/cue_meta_info.jsonl')
    cue_meta_info_dict[seed_id] = cue_meta_info
    
    
seed_cue_df = pd.read_csv(seed_cue_csv)

In [4]:
unique_seed_cue_lst = seed_cue_df['cue'].unique().tolist()
print(len(unique_seed_cue_lst))

1005


In [5]:
# also load test set cues
with open(meta_info_json_fp, 'r') as f:
    cue_meta_info = json.loads(f.read())
       

In [ ]:
print(cue_meta_info['cue_info']['test'][0])
print(unique_seed_cue_lst[0])

test_cues = [x for x in cue_meta_info['cue_info']['test']]
# replace Cue word: 
test_cues = [x.replace("Cue word: ", "") for x in test_cues]

# check overlap
overlap_cues = set(test_cues).intersection(set(unique_seed_cue_lst))
print(len(overlap_cues))

Cue word: cruelty
America
210


In [12]:
unique_seed_cue_lst.append('God')
unique_seed_cue_lst.append('industrialize')
unique_seed_cue_lst.append('asdfsdafadsfasdf')


In [13]:
# remove overlap cues from seed cues
unique_seed_cue_lst = [x for x in unique_seed_cue_lst if x not in overlap_cues]
print(len(unique_seed_cue_lst))

798


In [14]:
from tqdm.auto import tqdm
seed_cue_map_info = dict()
for seed_cue in tqdm(unique_seed_cue_lst, desc="Mapping seed cues to meta info"):
    find_flag = False
    for seed_id, cue_meta_info in cue_meta_info_dict.items():
        for locid, val in cue_meta_info.iterrows():
            for data_cue_word_dirty in val['cue_info']:
                clean_dataset_cue_word = data_cue_word_dirty.replace("Cue word: ", "")
                if clean_dataset_cue_word == seed_cue:
                    seed_cue_map_info[seed_cue] = {
                        'meta_seed_id': seed_id,
                        'meta_locid': locid,
                    }
                    find_flag = True
                    break
            if find_flag:
                break
        if find_flag:
            break
    if not find_flag:
        print(f"Warning: Could not find meta info for seed cue '{seed_cue}'")
                        

Mapping seed cues to meta info:   0%|          | 0/798 [00:00<?, ?it/s]

In [15]:
print(len(seed_cue_map_info))

795


In [16]:
# loop over seed_cue_map_info and get the dataset 
import datasets
cue_training_data_collection = [] 

# preprocess cue_info
dataset_seed_split_cue_map = dict()
for seed_cue, meta_info in tqdm(seed_cue_map_info.items(), desc="Generating datasets for seed cues"):
    seed_id = meta_info['meta_seed_id']
    locid = meta_info['meta_locid']
    if seed_id not in dataset_seed_split_cue_map:
        dataset_seed_split_cue_map[seed_id] = dict()
    if locid != 'test':
        split_name = 'train'
    else:
        split_name = 'test'
    if split_name not in dataset_seed_split_cue_map[seed_id]:
        dataset_seed_split_cue_map[seed_id][split_name] = []
    dataset_seed_split_cue_map[seed_id][split_name].append(seed_cue)
def filter_helper(example, cue_list):
    for cue in cue_list:
        if f"Cue word: {cue}" == example['input']: # ! not in, but exact match
            return True
    return False

for dataset_seed in [43,44,45,46,47]:
    the_dataset = datasets.load_dataset(root_dataset_dir,f"participant_swow_collection_us_seed_{dataset_seed}")
    for split_name in ['train', 'test']:
        target_dataset = the_dataset[split_name]
        if dataset_seed in dataset_seed_split_cue_map and split_name in dataset_seed_split_cue_map[dataset_seed]:
            cue_list = dataset_seed_split_cue_map[dataset_seed][split_name]
            filtered_data = target_dataset.filter(
                lambda x: filter_helper(x, cue_list),
                desc=f"Filtering seed {dataset_seed} split {split_name}"
            )
            cue_training_data_collection.extend(filtered_data)
        


print(f"Total collected training data samples: {len(cue_training_data_collection)}")
            

Generating datasets for seed cues:   0%|          | 0/795 [00:00<?, ?it/s]

Filtering seed 43 split test:   0%|          | 0/239389 [00:00<?, ? examples/s]

Total collected training data samples: 78798


In [17]:
cue_training_data_collection[0]

{'system': 'You are a human participant in a psychology experiment.',
 'instruction': '### Background ###\nOn average, an adult knows about 40,000 words, but what do these words mean to people like you and me? You can help scientists understand how meaning is organized in our mental dictionary by playing the game of word associations. This game is easy: Just give the first three words that come to mind for a given cue word.\n### OUTPUT FORMAT ###\nOutput your response in the following format:\nresponse1, response2, response3\nDo not provide any additional context, or explanations. Just the words as comma-separated values.\nIf you cannot think of further responses after response1 or response2, output the token NO MORE RESPONSE for the remaining slot(s).\n### End of instructions ###\n',
 'input': 'Cue word: last',
 'output': 'first, remainder, least'}

In [18]:
from pathlib import Path
seed_cue_dataset_name = f"seed_cue_participant_swow_collection_us"
seed_cue_dataset_save_fp = os.path.join(root_dataset_dir, seed_cue_dataset_name, 'train', 'chunk_0.jsonl')
Path(os.path.dirname(seed_cue_dataset_save_fp)).mkdir(parents=True, exist_ok=True)

import jsonlines

with jsonlines.open(seed_cue_dataset_save_fp, mode='w') as writer:
    for item in cue_training_data_collection:
        writer.write(item)

In [19]:
# ! finally  
# # try to load the saved dataset
# need to delete huggingface cache first
cache_location = os.path.join(os.environ['HF_HOME'], 'datasets', 'participant_swow_collection_us', f"seed_cue_participant_swow_collection_us")
print(cache_location)
import shutil
if os.path.exists(cache_location):
    shutil.rmtree(cache_location)

import datasets
test_loaded_dataset = datasets.load_dataset(root_dataset_dir, f"seed_cue_participant_swow_collection_us")
print(test_loaded_dataset)

/data/gpfs/projects/punim2219/LM_with_SWOW/sukaih/cultural-lexis-finetune-llms/../huggingface/datasets/participant_swow_collection_us/seed_cue_participant_swow_collection_us


Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['system', 'instruction', 'input', 'output'],
        num_rows: 78798
    })
})


In [20]:
# reset the unique number of cues
# find unique number of test_loaded_dataset['train']['input']
from tqdm.auto import tqdm
unique_input_set = set()
for input_text in tqdm(test_loaded_dataset['train']['input']):
    unique_input_set.add(input_text)

print(f"Unique number of input cues in the loaded dataset: {len(unique_input_set)}")

  0%|          | 0/78798 [00:00<?, ?it/s]

Unique number of input cues in the loaded dataset: 795


In [ ]:
# BELOW is other analysis

In [15]:
for seed_cue in unique_seed_cue_lst:
    find_flag = False
    for set_val in unique_input_set:
        if f"Cue word: {seed_cue}" in set_val:
            find_flag = True
            break
    if not find_flag:
        print(f"Seed cue '{seed_cue}' not found in the loaded dataset inputs.")

Seed cue 'industrialise' not found in the loaded dataset inputs.
Seed cue 'asdfsdafadsfasdf' not found in the loaded dataset inputs.


In [16]:
for seed_cue in unique_seed_cue_lst:
    find_flag = False
    find_extend_val = None 
    find_exact_match = False 
    for set_val in unique_input_set:
        if set_val == f"Cue word: {seed_cue}":
            find_exact_match = True
        if f"Cue word: {seed_cue}" in set_val:
            find_flag = True
            find_extend_val = set_val
            break
        
    if not find_flag:
        print(f"Seed cue '{seed_cue}' not found in the loaded dataset inputs.")
    if not find_exact_match:
        print(f"Seed cue '{seed_cue}' does not have an exact match in the loaded dataset inputs, closest match: '{find_extend_val}'")

Seed cue 'America' does not have an exact match in the loaded dataset inputs, closest match: 'Cue word: Americans'
Seed cue 'Christ' does not have an exact match in the loaded dataset inputs, closest match: 'Cue word: Christmas'
Seed cue 'a' does not have an exact match in the loaded dataset inputs, closest match: 'Cue word: aghast'
Seed cue 'account' does not have an exact match in the loaded dataset inputs, closest match: 'Cue word: accountant'
Seed cue 'add' does not have an exact match in the loaded dataset inputs, closest match: 'Cue word: additional'
Seed cue 'advertise' does not have an exact match in the loaded dataset inputs, closest match: 'Cue word: advertisement'
Seed cue 'again' does not have an exact match in the loaded dataset inputs, closest match: 'Cue word: against'
Seed cue 'age' does not have an exact match in the loaded dataset inputs, closest match: 'Cue word: agent'
Seed cue 'agree' does not have an exact match in the loaded dataset inputs, closest match: 'Cue wo